# What's Open: geocode and load businesses

Notebook version of the loader. It runs the same code as `py scripts/load_business_data.py`:
reads the workbook, copies its columns onto the map data, then places every business that has a **Display address**
with Olmsted County address points, falling back to Nominatim (OpenStreetMap), then the US Census geocoder.
`data/whats-open-data.js` and `index.html` are only written in step 4, after you have previewed the result.

Run the cells in order. Needs `pandas` and `openpyxl`; `folium` is optional (for the map).

In [ ]:
# 1. Setup: point Python at the repo's scripts folder
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts" / "load_business_data.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Open this notebook from inside the Whats_Open_Downtown_Rochester folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
import load_business_data as loader
print("Repo:", ROOT)

In [ ]:
# 2. Choose the workbook: a dialog opens; cancel it to use the newest workbook in data/
WORKBOOK = None
try:
    import tkinter
    from tkinter import filedialog
    root = tkinter.Tk(); root.withdraw(); root.call("wm", "attributes", ".", "-topmost", True)
    picked = filedialog.askopenfilename(title="Select the Business Inventory workbook",
                                        filetypes=[("Excel", "*.xlsx *.xlsm")], initialdir=ROOT / "data")
    WORKBOOK = Path(picked) if picked else None
except Exception as error:
    print("No file dialog available:", error)
WORKBOOK = WORKBOOK or loader.newest_workbook()
print("Workbook:", WORKBOOK)

sheet = pd.DataFrame(list(loader.read_sheet(WORKBOOK)))
print(len(sheet), "businesses;", int(sheet["address"].notna().sum()) if "address" in sheet else 0, "have a Display address")
sheet.head()

In [ ]:
# 3. Dry run: shows every change (including businesses that would be removed) and writes nothing
def run_loader(*flags):
    old = sys.argv
    sys.argv = ["load_business_data.py", str(WORKBOOK), *flags]
    try:
        loader.main()
    finally:
        sys.argv = old

run_loader("--dry-run")

In [ ]:
# 4. Write for real: updates data/whats-open-data.js and re-embeds it in index.html
#    Check the dry run first. The workbook is the complete list, so businesses missing from it are removed.
run_loader()

In [ ]:
# 5. Review the result: sources used and a map of the placed businesses
_, data = loader.load_data()
result = pd.DataFrame(data["businesses"])
print(result["geocodeStatus"].fillna("(not geocoded: no address)").value_counts().to_string())

try:
    import folium
    m = folium.Map(location=[result["lat"].mean(), result["lng"].mean()], zoom_start=16, width=900, height=550)
    for _, b in result.dropna(subset=["lat", "lng"]).iterrows():
        folium.CircleMarker([b["lat"], b["lng"]], radius=5, color="#005A70", fill=True,
                            tooltip=f"{b['name']} | {b.get('geocodeStatus') or 'existing coordinates'}").add_to(m)
    display(m)
except ImportError:
    print("Install folium (pip install folium) to see the map.")
    display(result[["name", "address", "lat", "lng", "geocodeStatus"]])